### Leetcode 184. Department Highest Salary

In [0]:
emp  = [(1,'Joe',70000,1),(2,'Jim',90000,1),(3,'Henry',80000,2),(4,'Sam',60000,2),(5,'Max',90000,1)]
dept = [(1,'IT'),(2,'Sales')]

Employee   = spark.createDataFrame(emp,  ['id','name','salary','departmentId'])
Department = spark.createDataFrame(dept, ['id','name'])


Employee
| id | name  | salary | departmentId |
|----|-------|--------|--------------|
| 1  | Joe   | 70000  | 1            |
| 2  | Jim   | 90000  | 1            |
| 3  | Henry | 80000  | 2            |
| 4  | Sam   | 60000  | 2            |
| 5  | Max   | 90000  | 1            |

Department
| id | name  |
|----|-------|
| 1  | IT    |
| 2  | Sales |

#### Spark SQL

In [0]:
spark.sql('''
          select d.name as Department, e.name as Employee, e.salary as Salary
from Employee e inner join Department d on e.departmentId = d.id
where e.salary = (select max(salary) from Employee 
where Employee.departmentId = e.departmentId
group by Employee.departmentId)
          ''').show()

#### Spark DataFrame API

In [0]:
from pyspark.sql import functions as F

from pyspark.sql.types import *

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "1")

max_salary_df = Employee.groupBy("departmentId").agg(F.max("salary").alias("max_salary"))

emp_with_max = Employee.join(max_salary_df, on="departmentId", how="inner")

result = emp_with_max.join(Department, emp_with_max.departmentId == Department.id)\
    .filter(emp_with_max.salary == emp_with_max.max_salary)\
    .select(
        Department.name.alias("Department"),
        emp_with_max.name.alias("Employee"),
        emp_with_max.salary.alias("Salary")
    )

result.show()